# Neural Network (PyTorch)

In [1]:
import time
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Visual configuration
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Deterministic CPU execution: reproducibility takes priority over GPU speed
# for this small dataset (216 samples).
DEVICE = torch.device('cpu')
torch.use_deterministic_algorithms(True)

## Carregamento e Exploração dos Dados

In [2]:
# Load dataset
df = pd.read_csv('../data/raw/hepg2.csv')
print(f'Dataset: {df.shape[0]} samples, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}\n')
df.describe().round(2)

Dataset: 216 samples, 6 columns
Columns: ['INDEX', 'ESPÉCIE/LINHAGEM', '% DMSO', 'TREHALOSE', '% SFB', 'VIABILIDADE']



,INDEX,% DMSO,TREHALOSE,% SFB,VIABILIDADE
count,216.00,216.00,216.00,216.00,216.00
mean,108.50,20.74,16.94,58.98,44.33
std,62.50,27.95,26.82,30.56,36.87
min,1.00,0.00,0.00,0.00,0.00
25%,54.75,2.00,0.00,40.00,0.00
50%,108.50,5.00,0.00,70.00,54.44
75%,162.25,30.00,30.00,81.25,76.78
max,216.00,100.00,100.00,100.00,97.33


In [3]:
# Check for linear dependency between variables
# SFB = 100 - DMSO - TREHALOSE (in most cases)
# Therefore, SFB is linearly dependent on DMSO and TREHALOSE.
# Including it as a feature would introduce multicollinearity without adding new information.
df['SUM'] = df['% DMSO'] + df['TREHALOSE'] + df['% SFB']
print('Sum of DMSO + TREHALOSE + SFB:')
print(df['SUM'].value_counts())
df.drop('SUM', axis=1, inplace=True)

Sum of DMSO + TREHALOSE + SFB:
SUM
100.0    208
10.0       8
Name: count, dtype: int64


In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[['% DMSO', 'TREHALOSE', '% SFB', 'VIABILIDADE']].corr()
corr = corr.rename(
    index={'% DMSO': 'DMSO', 'TREHALOSE': 'Trehalose', '% SFB': 'SFB', 'VIABILIDADE': 'Viability'},
    columns={'% DMSO': 'DMSO', 'TREHALOSE': 'Trehalose', '% SFB': 'SFB', 'VIABILIDADE': 'Viability'},
)
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.3f',
    ax=ax
)
ax.set_title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/eda_correlation_matrix.png', dpi=300)
plt.show()

In [ ]:
# Panels A (DMSO) and B (Trehalose): mean +/- SD per discrete experimental level,
# over individual (jittered) replicates. No parametric fit -- both cryoprotectants
# were tested at a small number of discrete levels (14 DMSO levels, 9 Trehalose
# levels) with unbalanced replicate counts, and DMSO's effect is a toxicity
# THRESHOLD, not a smooth curve. A polynomial fit here would extrapolate to
# biologically impossible negative viability and misrepresent the threshold.
rng = np.random.default_rng(SEED)


def plot_threshold_panel(ax, x_col, color, label, panel_letter):
    x = df[x_col].values.astype(float)
    y = df['VIABILIDADE'].values.astype(float)

    span = x.max() - x.min()
    jitter = rng.uniform(-1, 1, size=len(x)) * (span * 0.006)
    ax.scatter(x + jitter, y, alpha=0.35, s=28, color=color, edgecolor='none',
               label='Individual replicates', zorder=2)

    levels = np.sort(df[x_col].unique())
    means = np.array([df.loc[df[x_col] == lv, 'VIABILIDADE'].mean() for lv in levels])
    stds = np.array([df.loc[df[x_col] == lv, 'VIABILIDADE'].std() for lv in levels])

    ax.errorbar(levels, means, yerr=stds, fmt='o-', color='black', ecolor='black',
                elinewidth=1.5, capsize=4, markersize=6, markerfacecolor=color,
                markeredgecolor='black', linewidth=2, label='Mean ± SD', zorder=3)

    ax.set_xlabel(f'{label} (%)', fontweight='bold')
    ax.set_ylabel('Viability (%)', fontweight='bold')
    ax.set_ylim(bottom=-3)
    ax.text(-0.10, 1.08, panel_letter, transform=ax.transAxes, fontsize=18,
            fontweight='bold', va='top')
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
    n_levels = len(levels)
    print(f'  Panel {panel_letter} ({label}): {n_levels} discrete levels, '
          f'n per level = {[int((df[x_col]==lv).sum()) for lv in levels]}')


print('Building Figure 1 (2 panels: A=DMSO, B=Trehalose)...')
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
plot_threshold_panel(axes[0], '% DMSO', 'steelblue', 'DMSO concentration', 'A')
plot_threshold_panel(axes[1], 'TREHALOSE', 'coral', 'Trehalose concentration', 'B')
plt.tight_layout()
plt.savefig('../static/images/eda_concentration_response.png', dpi=300, bbox_inches='tight')
plt.show()

# Viability histogram, kept as a separate figure (previously the 3rd panel here)
print('\nBuilding viability histogram (separate file)...')
fig2, ax2 = plt.subplots(figsize=(7, 5))
ax2.hist(df['VIABILIDADE'], bins=30, color='seagreen', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Viability (%)', fontweight='bold')
ax2.set_ylabel('Frequency', fontweight='bold')
ax2.set_title('Viability Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/eda_viability_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

## Feature Importance

In [6]:
# Original features
X_raw = df[['% DMSO', 'TREHALOSE']].values
y_all = df['VIABILIDADE'].values
# Feature importance using Random Forest
rf_temp = RandomForestRegressor(
    n_estimators=200,
    random_state=SEED
)
rf_temp.fit(X_raw, y_all)
print('Feature Importance:')
for name, imp in zip(['% DMSO', 'TREHALOSE'], rf_temp.feature_importances_):
    print(f'  {name}: {imp:.4f}')

Feature Importance:
  % DMSO: 0.7662
  TREHALOSE: 0.2338


## Data Preparation

In [ ]:
# =============================================================
# Triple split: 60% train / 20% validation / 20% test
# =============================================================
# The VALIDATION set is used for Early Stopping and LR Scheduler.
# The TEST set is held out and used ONLY ONCE for final evaluation.
# Raw features only: [% DMSO, TREHALOSE]. The polynomial expansion used
# previously (DMSO^2, DMSO x Trehalose, Trehalose^2) has been removed --
# the hyperparameter search below shows the network matches the old
# polynomial-feature performance once regularization is tuned correctly.
# =============================================================

# Step 1: split into 80% (train+validation) and 20% (test)
X_trainval_raw, X_test_raw, y_trainval, y_test = train_test_split(
    X_raw,
    y_all,
    test_size=0.2,
    random_state=SEED
)

# Step 2: split the 80% into 60% training and 20% validation
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_trainval_raw,
    y_trainval,
    test_size=0.25,
    random_state=SEED
)

# Standardization (fit ONLY on the 129-sample fit set)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

print(f'Train (fit): {X_train_scaled.shape[0]} samples, {X_train_scaled.shape[1]} features')
print(f'Validation:  {X_val_scaled.shape[0]} samples')
print(f'Test:        {X_test_scaled.shape[0]} samples')
print(f'Total:       {X_train_scaled.shape[0] + X_val_scaled.shape[0] + X_test_scaled.shape[0]} samples')

## Neural Network Architecture

* **Input layer**: 2 neurons -- raw `% DMSO` and `TREHALOSE` only (no polynomial expansion; polynomial features were found to add no benefit once regularization was tuned correctly -- see the hyperparameter search below)
* **3 hidden layers**: 128 → 64 → 32 neurons
* **Batch Normalization** and **Dropout**: both are now *searched* hyperparameters (see below), not fixed choices
* **ReLU activation** in the hidden layers
* **Output layer**: 1 neuron (predicted cell viability, %)

In [8]:
class HepatoCryoNN(nn.Module):
    """Flexible architecture: BatchNorm and Dropout are toggled by config.
    Everything else (128 -> 64 -> 32 -> 1, ReLU) is fixed."""

    def __init__(self, input_dim=2, use_bn=True, dropout_rate=0.0):
        super().__init__()
        dims = [input_dim, 128, 64, 32]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if use_bn:
                layers.append(nn.BatchNorm1d(dims[i + 1]))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
        layers.append(nn.Linear(32, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def train_model(X_tr, y_tr, X_vl, y_vl, input_dim, use_bn, dropout, weight_decay,
                 patience, seed, max_epochs=2000, track_history=False, log_every=None):
    """Train one model instance with early stopping on VALIDATION loss.

    Fixed across all configurations: architecture 128->64->32->1, ReLU,
    Adam(lr=1e-3), ReduceLROnPlateau(factor=0.5, patience=30, min_lr=1e-6),
    MSE loss, batch_size=32, CPU, deterministic.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_tr_t = torch.FloatTensor(X_tr).to(DEVICE)
    y_tr_t = torch.FloatTensor(y_tr).unsqueeze(1).to(DEVICE)
    X_vl_t = torch.FloatTensor(X_vl).to(DEVICE)
    y_vl_t = torch.FloatTensor(y_vl).unsqueeze(1).to(DEVICE)

    gen = torch.Generator()
    gen.manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t),
        batch_size=32, shuffle=True, drop_last=True, generator=gen
    )

    model = HepatoCryoNN(input_dim=input_dim, use_bn=use_bn, dropout_rate=dropout).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=30, min_lr=1e-6
    )

    best_val_loss = float('inf')
    best_epoch = 0
    no_improve = 0
    best_state = None
    train_losses, val_losses = [], []

    for epoch in range(max_epochs):
        model.train()
        epoch_train_loss = 0.0
        n_batches = 0
        for X_b, y_b in loader:
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item()
            n_batches += 1

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_vl_t), y_vl_t).item()
        scheduler.step(val_loss)

        if track_history:
            train_losses.append(epoch_train_loss / max(n_batches, 1))
            val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1

        if log_every and (epoch + 1) % log_every == 0:
            print(f'    epoch {epoch+1:4d} | train_loss={epoch_train_loss/max(n_batches,1):.4f} '
                  f'| val_loss={val_loss:.4f} | lr={optimizer.param_groups[0]["lr"]:.6f}')

        if no_improve >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    return model, best_epoch, best_val_loss, train_losses, val_losses


print('HepatoCryoNN and train_model() defined. Architecture: 2 -> 128 -> 64 -> 32 -> 1 (ReLU); '
      'batch_norm, dropout, weight_decay, patience are search hyperparameters (see below).')

HepatoCryoNN and train_model() defined. Architecture: 2 -> 128 -> 64 -> 32 -> 1 (ReLU); batch_norm, dropout, weight_decay, patience are search hyperparameters (see below).


## Hyperparameter Search (selected by VALIDATION loss only)

To avoid data leakage, the regularization configuration is chosen using the mean
**validation** MSE across 3 seeds -- the test set is never used for model selection.

Grid: `batch_norm` ∈ {True, False} × `dropout` ∈ {0.0, 0.1, 0.2} × `weight_decay` ∈ {0, 1e-4}
× `patience` ∈ {100, 200} = 24 configurations, 3 seeds each (0, 1, 42) = 72 runs.

Everything else is fixed: architecture 128→64→32→1, ReLU, Adam(lr=1e-3),
`ReduceLROnPlateau(factor=0.5, patience=30, min_lr=1e-6)`, MSE loss, batch_size=32,
max 2000 epochs, early stopping on validation loss.

In [9]:
# ============================================================
# Small hyperparameter search, 3 seeds per configuration.
# Selection criterion: mean VALIDATION MSE across seeds (test set untouched).
# ============================================================
grid = list(itertools.product(
    [True, False],       # batch_norm
    [0.0, 0.1, 0.2],      # dropout
    [0.0, 1e-4],          # weight_decay
    [100, 200],           # patience
))
search_seeds = [0, 1, 42]

search_rows = []
t_search_start = time.time()

for gi, (use_bn, dropout, wd, patience) in enumerate(grid):
    for seed in search_seeds:
        t0 = time.time()
        _, best_epoch_s, best_val_loss_s, _, _ = train_model(
            X_train_scaled, y_train, X_val_scaled, y_val, input_dim=2,
            use_bn=use_bn, dropout=dropout, weight_decay=wd, patience=patience, seed=seed
        )
        elapsed = time.time() - t0
        search_rows.append({
            'config_id': gi, 'batch_norm': use_bn, 'dropout': dropout,
            'weight_decay': wd, 'patience': patience, 'seed': seed,
            'best_epoch': best_epoch_s, 'val_loss': best_val_loss_s, 'time_s': elapsed
        })
        print(f'[{gi+1}/{len(grid)}] bn={use_bn} drop={dropout} wd={wd} pat={patience} '
              f'seed={seed} -> val_loss={best_val_loss_s:.4f} ep={best_epoch_s} ({elapsed:.1f}s) '
              f'total_elapsed={time.time()-t_search_start:.0f}s')

search_df = pd.DataFrame(search_rows)

search_summary = search_df.groupby(
    ['config_id', 'batch_norm', 'dropout', 'weight_decay', 'patience']
).agg(
    mean_val_loss=('val_loss', 'mean'),
    std_val_loss=('val_loss', 'std'),
    mean_best_epoch=('best_epoch', 'mean'),
).reset_index().sort_values('mean_val_loss')

print('\n=== SEARCH SUMMARY (sorted by mean val loss) ===')
print(search_summary.to_string(index=False))

winner = search_summary.iloc[0]
print('\nWINNER:', winner.to_dict())
print(f'\nTotal search time: {time.time()-t_search_start:.0f}s')

[1/24] bn=True drop=0.0 wd=0.0 pat=100 seed=0 -> val_loss=60.9776 ep=322 (10.9s) total_elapsed=11s
[1/24] bn=True drop=0.0 wd=0.0 pat=100 seed=1 -> val_loss=66.2938 ep=333 (8.1s) total_elapsed=19s
[1/24] bn=True drop=0.0 wd=0.0 pat=100 seed=42 -> val_loss=63.0138 ep=586 (14.7s) total_elapsed=34s
[2/24] bn=True drop=0.0 wd=0.0 pat=200 seed=0 -> val_loss=60.9776 ep=322 (11.3s) total_elapsed=45s
[2/24] bn=True drop=0.0 wd=0.0 pat=200 seed=1 -> val_loss=63.0080 ep=636 (17.7s) total_elapsed=63s
[2/24] bn=True drop=0.0 wd=0.0 pat=200 seed=42 -> val_loss=63.0138 ep=586 (16.8s) total_elapsed=79s
[3/24] bn=True drop=0.0 wd=0.0001 pat=100 seed=0 -> val_loss=59.0692 ep=314 (9.7s) total_elapsed=89s
[3/24] bn=True drop=0.0 wd=0.0001 pat=100 seed=1 -> val_loss=58.0063 ep=497 (15.2s) total_elapsed=104s
[3/24] bn=True drop=0.0 wd=0.0001 pat=100 seed=42 -> val_loss=72.2519 ep=365 (10.3s) total_elapsed=115s
[4/24] bn=True drop=0.0 wd=0.0001 pat=200 seed=0 -> val_loss=57.7734 ep=587 (14.6s) total_elapsed

## Final Model Training (winning configuration, seed=42)

In [10]:
WINNING_CONFIG = dict(
    use_bn=bool(winner['batch_norm']),
    dropout=float(winner['dropout']),
    weight_decay=float(winner['weight_decay']),
    patience=int(winner['patience']),
)
print(f'Training final model with winning configuration: {WINNING_CONFIG}\n')

final_model, best_epoch, best_val_loss, train_losses, val_losses = train_model(
    X_train_scaled, y_train, X_val_scaled, y_val, input_dim=2,
    use_bn=WINNING_CONFIG['use_bn'], dropout=WINNING_CONFIG['dropout'],
    weight_decay=WINNING_CONFIG['weight_decay'], patience=WINNING_CONFIG['patience'],
    seed=42, track_history=True, log_every=100
)

print(f'\nFinal training completed. Best epoch: {best_epoch} (Val Loss: {best_val_loss:.4f})')

Training final model with winning configuration: {'use_bn': False, 'dropout': 0.0, 'weight_decay': 0.0001, 'patience': 100}

    epoch  100 | train_loss=541.1666 | val_loss=224.5128 | lr=0.001000
    epoch  200 | train_loss=306.0403 | val_loss=111.6641 | lr=0.001000
    epoch  300 | train_loss=132.7829 | val_loss=66.3637 | lr=0.001000
    epoch  400 | train_loss=33.5430 | val_loss=49.9537 | lr=0.001000
    epoch  500 | train_loss=28.8631 | val_loss=45.7519 | lr=0.000250

Final training completed. Best epoch: 441 (Val Loss: 43.3481)


In [11]:
print('Best model (from validation-based early stopping) is already loaded into `final_model`.')
print(f'Winning configuration: {WINNING_CONFIG}')
print(f'Best epoch: {best_epoch}')

Best model (from validation-based early stopping) is already loaded into `final_model`.
Winning configuration: {'use_bn': False, 'dropout': 0.0, 'weight_decay': 0.0001, 'patience': 100}
Best epoch: 441


## Evaluation: Fit / Validation / Test Sets

In [12]:
# Final predictions (the test set is touched for the FIRST and ONLY time here)
final_model.eval()
with torch.no_grad():
    y_pred_train = final_model(torch.FloatTensor(X_train_scaled)).numpy().flatten()
    y_pred_val   = final_model(torch.FloatTensor(X_val_scaled)).numpy().flatten()
    y_pred_test  = final_model(torch.FloatTensor(X_test_scaled)).numpy().flatten()

r2_train = r2_score(y_train, y_pred_train)
r2_val   = r2_score(y_val,   y_pred_val)
r2_test  = r2_score(y_test,  y_pred_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_val   = np.sqrt(mean_squared_error(y_val,   y_pred_val))
rmse_test  = np.sqrt(mean_squared_error(y_test,  y_pred_test))

residuals_test = y_test - y_pred_test
resid_mean = residuals_test.mean()
resid_std  = residuals_test.std()

print('PERFORMANCE METRICS — NEURAL NETWORK (PyTorch, raw features)\n')
print(f'Fit (129)        -> R² = {r2_train:.4f} | RMSE = {rmse_train:.4f}%')
print(f'Validation (43)  -> R² = {r2_val:.4f} | RMSE = {rmse_val:.4f}%')
print(f'Test (44)        -> R² = {r2_test:.4f} | RMSE = {rmse_test:.4f}%')
print(f'\nBest epoch (early stopping): {best_epoch}')
print(f'Test residuals   -> mean = {resid_mean:.4f} | std = {resid_std:.4f}')

if r2_test < 0.85:
    print('\n*** WARNING: test R² below the 0.85 sanity threshold. Investigate before concluding. ***')

PERFORMANCE METRICS — NEURAL NETWORK (PyTorch, raw features)

Fit (129)        -> R² = 0.9795 | RMSE = 5.4652%
Validation (43)  -> R² = 0.9576 | RMSE = 6.5839%
Test (44)        -> R² = 0.9788 | RMSE = 5.1370%

Best epoch (early stopping): 441
Test residuals   -> mean = 0.3362 | std = 5.1260


## View

In [ ]:
# Loss Curve (Training vs Validation)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_losses, label='Train Loss', color='#1f77b4', alpha=0.7)
ax.plot(val_losses, label='Validation Loss', color='#d62728', alpha=0.7)
ax.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.5, label=f'Best epoch ({best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Learning Curve — Neural Network (ANN)', fontweight='bold')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('../static/images/nn_loss_curve.png', dpi=300)
plt.show()

In [ ]:
# Plot: Actual vs Predicted
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_test, y_pred_test, alpha=0.7, color='#d62728', s=80, 
           edgecolors='black', linewidths=0.5, label='Test Data', zorder=3)
ax.scatter(y_train, y_pred_train, alpha=0.3, color='#1f77b4', s=40, 
           label='Train Data', zorder=2)

lims = [min(y_all.min(), 0), max(y_all.max(), 100)]
ax.plot(lims, lims, 'k--', lw=2, label='Perfect Prediction', zorder=1)

ax.set_xlabel('Observed in vitro Viability (%)', fontweight='bold')
ax.set_ylabel('Predicted in silico Viability (%)', fontweight='bold')
ax.set_title(f'Neural Network (ANN) — Actual vs Predicted (R² = {r2_test:.4f})', fontweight='bold', fontsize=14)
ax.legend()
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('../static/images/nn_actual_vs_predicted.png', dpi=300)
plt.show()

In [ ]:
# Residuals Plot
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_test, residuals, alpha=0.7, color='#d62728', s=60, edgecolors='black', linewidths=0.5)
axes[0].axhline(y=0, color='black', linestyle='--', lw=1.5)
axes[0].set_xlabel('Predicted Viability (%)')
axes[0].set_ylabel('Residual (%)')
axes[0].set_title('Residuals vs. Predicted', fontweight='bold')

# Distribution of Residuals
axes[1].hist(residuals, bins=20, color='#d62728', edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='black', linestyle='--', lw=1.5)
axes[1].set_xlabel('Residual (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residuals Distribution (μ={residuals.mean():.4f}, σ={residuals.std():.4f})', fontweight='bold')

plt.tight_layout()
plt.savefig('../static/images/nn_residuals.png', dpi=300)
plt.show()

## K-Fold Cross-Validation (K=5)

Runs over the 172 train+validation samples only (the 44-sample test set stays
untouched) using the winning configuration from the search above. The
`StandardScaler` is refit inside each fold to avoid leakage.

In [16]:
def train_and_evaluate_fold(X_train_f, y_train_f, X_val_f, y_val_f, input_dim, config, seed=SEED):
    """Train one fold of the neural network (winning configuration) and return its validation metrics."""
    fold_model, fold_best_epoch, fold_best_val_loss, _, _ = train_model(
        X_train_f, y_train_f, X_val_f, y_val_f, input_dim=input_dim,
        use_bn=config['use_bn'], dropout=config['dropout'],
        weight_decay=config['weight_decay'], patience=config['patience'], seed=seed
    )
    fold_model.eval()
    with torch.no_grad():
        y_pred_val = fold_model(torch.FloatTensor(X_val_f)).numpy().flatten()
    r2 = r2_score(y_val_f, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val_f, y_pred_val))
    return r2, rmse, fold_best_epoch


# ============================================================
# 5-fold CV over the 172 train+val samples (test set stays untouched).
# StandardScaler is refit INSIDE each fold to avoid leakage.
# ============================================================
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

fold_r2 = []
fold_rmse = []

print('--- K-FOLD CROSS-VALIDATION (K=5, winning configuration) ---\n')

for fold, (train_idx, val_idx) in enumerate(kf.split(X_trainval_raw)):

    X_tr_raw_f = X_trainval_raw[train_idx]
    X_vl_raw_f = X_trainval_raw[val_idx]

    y_tr_f = y_trainval[train_idx]
    y_vl_f = y_trainval[val_idx]

    # Standardization refit on this fold's training data only
    sc_f = StandardScaler()
    X_tr_sc_f = sc_f.fit_transform(X_tr_raw_f)
    X_vl_sc_f = sc_f.transform(X_vl_raw_f)

    r2, rmse, ep = train_and_evaluate_fold(
        X_tr_sc_f, y_tr_f, X_vl_sc_f, y_vl_f, input_dim=2, config=WINNING_CONFIG
    )

    fold_r2.append(r2)
    fold_rmse.append(rmse)

    print(f'  Fold {fold + 1}: R² = {r2:.4f} | RMSE = {rmse:.4f} | best_epoch = {ep}')

print(f'\n  MEAN: R² = {np.mean(fold_r2):.4f} (+/- {np.std(fold_r2):.4f})')
print(f'  MEAN: RMSE = {np.mean(fold_rmse):.4f} (+/- {np.std(fold_rmse):.4f})')

--- K-FOLD CROSS-VALIDATION (K=5, winning configuration) ---

  Fold 1: R² = 0.9486 | RMSE = 7.1297 | best_epoch = 542
  Fold 2: R² = 0.9611 | RMSE = 7.5270 | best_epoch = 747
  Fold 3: R² = 0.9856 | RMSE = 4.7609 | best_epoch = 707
  Fold 4: R² = 0.9718 | RMSE = 6.0928 | best_epoch = 534
  Fold 5: R² = 0.9728 | RMSE = 6.0257 | best_epoch = 634

  MEAN: R² = 0.9680 (+/- 0.0124)
  MEAN: RMSE = 6.3072 (+/- 0.9676)


## Seed-Stability Check (control only — not a headline result)

Retrains the final configuration with 5 different seeds (0, 1, 2, 42, 123) and
reports the mean ± std of the test R². This is a robustness control, not part
of the paper's primary result.

In [17]:
stability_seeds = [0, 1, 2, 42, 123]
stability_r2 = []

print('--- SEED-STABILITY CHECK ---\n')

for s in stability_seeds:
    m_s, ep_s, vl_s, _, _ = train_model(
        X_train_scaled, y_train, X_val_scaled, y_val, input_dim=2,
        use_bn=WINNING_CONFIG['use_bn'], dropout=WINNING_CONFIG['dropout'],
        weight_decay=WINNING_CONFIG['weight_decay'], patience=WINNING_CONFIG['patience'],
        seed=s
    )
    m_s.eval()
    with torch.no_grad():
        y_pred_s = m_s(torch.FloatTensor(X_test_scaled)).numpy().flatten()
    r2_s = r2_score(y_test, y_pred_s)
    stability_r2.append(r2_s)
    print(f'  seed={s:4d} -> test R² = {r2_s:.4f} (best_epoch={ep_s})')

stability_mean = np.mean(stability_r2)
stability_std = np.std(stability_r2)

print(f'\nTest R² across seeds: {stability_mean:.4f} +/- {stability_std:.4f}')

if stability_std > 0.05:
    print(f'*** WARNING: seed std ({stability_std:.4f}) > 0.05 -- result is SENSITIVE TO SEED. ***')
else:
    print(f'Seed std ({stability_std:.4f}) <= 0.05 -- result is reasonably stable across seeds.')

--- SEED-STABILITY CHECK ---

  seed=   0 -> test R² = 0.9774 (best_epoch=457)
  seed=   1 -> test R² = 0.9779 (best_epoch=497)
  seed=   2 -> test R² = 0.9734 (best_epoch=520)
  seed=  42 -> test R² = 0.9788 (best_epoch=441)
  seed= 123 -> test R² = 0.9778 (best_epoch=659)

Test R² across seeds: 0.9771 +/- 0.0019
Seed std (0.0019) <= 0.05 -- result is reasonably stable across seeds.


## Final Comparison: Random Forest vs. Neural Network

In [18]:
# Random Forest trained under the same conditions
# (using only raw features, with 60% training data)

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=SEED
)

rf_model.fit(X_train_raw, y_train)

y_pred_rf = rf_model.predict(X_test_raw)

r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f'Random Forest -> R² = {r2_rf:.4f} | RMSE = {rmse_rf:.2f}%')
print(f'Neural Network -> R² = {r2_test:.4f} | RMSE = {rmse_test:.2f}%')

Random Forest -> R² = 0.9745 | RMSE = 5.63%
Neural Network -> R² = 0.9788 | RMSE = 5.14%


In [ ]:
# Comparative plot: side-by-side scatter plots

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
lims = [0, 105]

# Random Forest
axes[0].scatter(
    y_test,
    y_pred_rf,
    alpha=0.7,
    color='#2ca02c',
    s=80,
    edgecolors='black',
    linewidths=0.5
)
axes[0].plot(lims, lims, 'k--', lw=2)
axes[0].set_xlabel('Observed Viability (%)')
axes[0].set_ylabel('Predicted Viability (%)')
axes[0].set_title(
    f'Random Forest (R² = {r2_rf:.3f})',
    fontweight='bold',
    fontsize=13
)

axes[0].set_xlim(lims)
axes[0].set_ylim(lims)
axes[0].set_aspect('equal')

# Neural Network
axes[1].scatter(
    y_test,
    y_pred_test,
    alpha=0.7,
    color="#d62728",
    s=80,
    edgecolors='black',
    linewidths=0.5
)

axes[1].plot(lims, lims, 'k--', lw=2)
axes[1].set_xlabel('Observed Viability (%)')
axes[1].set_ylabel('Predicted Viability (%)')
axes[1].set_title(
    f'Neural Network (ANN) (R² = {r2_test:.3f})',
    fontweight='bold',
    fontsize=13
)

axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_aspect('equal')

plt.suptitle(
    'Comparison: Random Forest vs. Neural Network (ANN)',
    fontsize=15,
    fontweight='bold',
    y=1.02
)

plt.tight_layout()
plt.savefig('../static/images/rf_vs_nn_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Model Export

Two artifacts are produced:
1. `../models/nn_model.pth` -- the trained PyTorch weights (kept for reproducibility/provenance).
2. `../models/nn_weights.npz` -- everything needed for **NumPy-only** inference in production: the
   `StandardScaler` parameters and the network's `Linear` weights. The winning configuration
   has no BatchNorm, so no folding is needed here; the export code below handles BatchNorm
   folding generically (Linear+BatchNorm → a single affine Linear layer) in case a future
   search picks a BatchNorm configuration.

`../src/nn_inference.py` implements `predict(dmso, trehalose)` using only NumPy and is validated
below against the PyTorch model on a dense concentration grid.

In [ ]:
def fold_bn_into_linear(linear, bn):
    """Fuse Linear + eval-mode BatchNorm1d into a single affine Linear transform."""
    W = linear.weight.detach().numpy()
    b = linear.bias.detach().numpy()
    gamma = bn.weight.detach().numpy()
    beta = bn.bias.detach().numpy()
    running_mean = bn.running_mean.detach().numpy()
    running_var = bn.running_var.detach().numpy()
    eps = bn.eps
    scale = gamma / np.sqrt(running_var + eps)
    return W * scale[:, None], scale * (b - running_mean) + beta


def export_layers(model):
    """Walk the Sequential, folding any Linear+BatchNorm pair; Dropout/ReLU are
    identity/structural in eval mode and are not stored."""
    modules = list(model.network)
    layers = []
    i, n = 0, len(modules)
    while i < n:
        mod = modules[i]
        if isinstance(mod, nn.Linear):
            if i + 1 < n and isinstance(modules[i + 1], nn.BatchNorm1d):
                W, b = fold_bn_into_linear(mod, modules[i + 1])
                i += 2
            else:
                W = mod.weight.detach().numpy()
                b = mod.bias.detach().numpy()
                i += 1
            layers.append((W.astype(np.float64), b.astype(np.float64)))
        else:
            i += 1
    return layers


exported_layers = export_layers(final_model)
print(f'Exported {len(exported_layers)} Linear(+ReLU) layers:')
for i, (W, b) in enumerate(exported_layers):
    print(f'  Layer {i}: W{W.shape} b{b.shape}')

npz_kwargs = {
    'mean': scaler.mean_.astype(np.float64),
    'scale': scaler.scale_.astype(np.float64),
    'n_layers': len(exported_layers),
}
for i, (W, b) in enumerate(exported_layers):
    npz_kwargs[f'W{i}'] = W
    npz_kwargs[f'b{i}'] = b

torch.save(final_model.state_dict(), '../models/nn_model.pth')
np.savez('../models/nn_weights.npz', **npz_kwargs)

print('\nFiles saved:')
print('  - ../models/nn_model.pth   (PyTorch weights, reproducibility/provenance)')
print('  - ../models/nn_weights.npz (StandardScaler + Linear weights, NumPy-only inference)')

## Acceptance Test — PyTorch vs. NumPy Inference

`src/nn_inference.py` must reproduce the PyTorch model's predictions using only NumPy.
This validates the exported artifact on a dense grid (DMSO 0-100, Trehalose 0-100,
step 1%, respecting DMSO + Trehalose ≤ 100). Required: max absolute difference < 1e-4.

In [ ]:
import sys
sys.path.insert(0, '../src')
import importlib
import nn_inference
importlib.reload(nn_inference)

dmso_vals = np.arange(0, 101, 1)
treh_vals = np.arange(0, 101, 1)
grid = [(d, t) for d, t in itertools.product(dmso_vals, treh_vals) if d + t <= 100]
grid_arr = np.array(grid, dtype=np.float64)
print(f'Grid size: {len(grid)} points (DMSO 0-100, Trehalose 0-100, step 1, sum <= 100)')

grid_scaled = scaler.transform(grid_arr)
final_model.eval()
with torch.no_grad():
    y_torch = final_model(torch.FloatTensor(grid_scaled)).numpy().flatten()

y_numpy = np.array([nn_inference.predict(d, t) for d, t in grid])

max_abs_diff = float(np.max(np.abs(y_torch - y_numpy)))
print(f'Max abs diff (PyTorch vs NumPy) over {len(grid)} grid points: {max_abs_diff:.3e}')

if max_abs_diff < 1e-4:
    print('PASS: max abs diff < 1e-4')
else:
    raise AssertionError('NumPy export does not match PyTorch within 1e-4 -- fix before proceeding!')

example_dmso, example_treh = 5.0, 10.0
example_torch = float(final_model(torch.FloatTensor(scaler.transform([[example_dmso, example_treh]]))).item())
example_numpy = nn_inference.predict(example_dmso, example_treh)
print(f'\nExample {example_dmso:.0f}% DMSO / {example_treh:.0f}% Trehalose -> '
      f'PyTorch = {example_torch:.4f} | NumPy = {example_numpy:.4f}')